# **Kaggle – DataTops®**
Tu TA ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Aspectos importantes
- Última submission:
    - Mañana: 17 de febrero a las 5pm
    - Tarde: 19 de febrero a las 5pm
- **Enlace de la competición**: https://www.kaggle.com/t/c5cc87b50c4b4770bdc8f5acbe15577d
- **Requisito**: Estar registrado en [Kaggle](https://www.kaggle.com/)

## Métrica:
El error cuadrático medio (RMSE, por sus siglas en inglés) es una medida de la desviación estándar de los residuos (errores de predicción). Los residuos representan la diferencia entre los valores observados y los valores predichos por el modelo. El RMSE indica qué tan dispersos están estos errores: cuanto menor es el RMSE, más cercanas están las predicciones a los valores reales. En otras palabras, el RMSE mide qué tan bien se ajusta la línea de regresión a los datos.


$$ RMSE = \sqrt{\frac{1}{n}\Sigma_{i=1}^{n}{\Big(\frac{d_i -f_i}{\sigma_i}\Big)^2}}$$


## 1. Librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from PIL import Image
import urllib.request

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import shap

import re

In [ ]:
pd.set_option('display.max_columns', 500)

## 2. Datos

In [ ]:
# Para que funcione necesitas bajarte los archivos de datos de Kaggle
df = pd.read_csv("./data/train.csv", index_col=0)

### 2.1 Exploración de los datos

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 912 entries, 755 to 229
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           912 non-null    object 
 1   Product           912 non-null    object 
 2   TypeName          912 non-null    object 
 3   Inches            912 non-null    float64
 4   ScreenResolution  912 non-null    object 
 5   Cpu               912 non-null    object 
 6   Ram               912 non-null    object 
 7   Memory            912 non-null    object 
 8   Gpu               912 non-null    object 
 9   OpSys             912 non-null    object 
 10  Weight            912 non-null    object 
 11  Price_in_euros    912 non-null    float64
dtypes: float64(2), object(10)
memory usage: 92.6+ KB


In [ ]:
df.head()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
laptop_ID,,,,,,,,,,,,
755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00
618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01
909,HP,ProBook 450,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00
2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
286,Dell,Inspiron 3567,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00


In [ ]:
df.tail()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
laptop_ID,,,,,,,,,,,,
28,Dell,Inspiron 5570,Notebook,15.6,Full HD 1920x1080,Intel Core i5 8250U 1.6GHz,8GB,256GB SSD,AMD Radeon 530,Windows 10,2.2kg,800.00
1160,HP,Spectre Pro,2 in 1 Convertible,13.3,Full HD / Touchscreen 1920x1080,Intel Core i5 6300U 2.4GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.48kg,1629.00
78,Lenovo,IdeaPad 320-15IKBN,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,2TB HDD,Intel HD Graphics 620,No OS,2.2kg,519.00
23,HP,255 G6,Notebook,15.6,1366x768,AMD E-Series E2-9000e 1.5GHz,4GB,500GB HDD,AMD Radeon R2,No OS,1.86kg,258.00
229,Dell,Alienware 17,Gaming,17.3,IPS Panel Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,256GB SSD + 1TB HDD,Nvidia GeForce GTX 1060,Windows 10,4.42kg,2456.34


In [ ]:
df.describe()

,Inches,Price_in_euros
count,912.000000,912.000000
mean,14.981579,1111.724090
std,1.436719,687.959172
min,10.100000,174.000000
25%,14.000000,589.000000
50%,15.600000,978.000000
75%,15.600000,1483.942500
max,18.400000,6099.000000


In [ ]:
for col in df.columns:
    print(col, '->', df[col].nunique())

Company -> 19
Product -> 480
TypeName -> 6
Inches -> 17
ScreenResolution -> 36
Cpu -> 107
Ram -> 9
Memory -> 37
Gpu -> 93
OpSys -> 9
Weight -> 165
Price_in_euros -> 603


In [ ]:
df['Company'].value_counts()

Company
Lenovo       202
Dell         197
HP           194
Asus         121
Acer          74
MSI           37
Toshiba       34
Apple         17
Razer          6
Mediacom       6
Samsung        5
Microsoft      5
Xiaomi         3
Huawei         2
Chuwi          2
Google         2
Vero           2
Fujitsu        2
LG             1
Name: count, dtype: int64

In [ ]:
brand_map = {
    'Apple': 10,
    'Razer': 9,
    'Microsoft': 8,
    'Lenovo': 7,
    'Dell': 6,
    'HP': 5,
    'Asus': 4,
    'Acer': 3,
    'MSI': 3,
    'Toshiba': 2,
    'Samsung': 2,
    'Huawei': 2,
    'Xiaomi': 1,
    'Chuwi': 1,
    'Google': 1,
    'Vero': 1,
    'Fujitsu': 1,
    'LG': 1,
    'Mediacom': 1
}

df['company_ordinal'] = df['Company'].map(brand_map).fillna(6)

df['company_ordinal'].value_counts()

company_ordinal
7     202
6     197
5     194
4     121
3     111
2      41
1      18
10     17
9       6
8       5
Name: count, dtype: int64

In [ ]:
df['TypeName'].value_counts()

TypeName
Notebook              509
Gaming                143
Ultrabook             141
2 in 1 Convertible     80
Workstation            20
Netbook                19
Name: count, dtype: int64

In [ ]:
type_map = {
    'Workstation': 6,
    'Gaming': 5,
    'Ultrabook': 4,
    'Notebook': 3,
    '2 in 1 Convertible': 2,
    'Netbook': 1
}

df['type_ordinal'] = df['TypeName'].map(type_map).fillna(3)
df['type_ordinal'].value_counts()

type_ordinal
3    509
5    143
4    141
2     80
6     20
1     19
Name: count, dtype: int64

In [ ]:
df['Product'].value_counts()

Product
XPS 13                                   23
Inspiron 3567                            22
Legion Y520-15IKBN                       15
Vostro 3568                              14
ProBook 450                              13
                                         ..
Inspiron 7773                             1
ENVY -                                    1
Latitude E7270                            1
Rog GL552VW-CN470T                        1
15-BS026nv (i5-7200U/8GB/256GB/Radeon     1
Name: count, Length: 480, dtype: int64

In [ ]:
df['product_len'] = df['Product'].str.len()

In [ ]:
df['Inches'].value_counts()

Inches
15.6    453
14.0    150
13.3    114
17.3    113
11.6     28
12.5     27
13.5      5
12.0      4
15.0      3
15.4      3
13.9      3
10.1      2
13.0      2
12.3      2
14.1      1
11.3      1
18.4      1
Name: count, dtype: int64

In [ ]:
df['ScreenResolution'].value_counts()

ScreenResolution
Full HD 1920x1080                                349
1366x768                                         211
IPS Panel Full HD 1920x1080                      163
IPS Panel Full HD / Touchscreen 1920x1080         32
Full HD / Touchscreen 1920x1080                   30
1600x900                                          14
Quad HD+ / Touchscreen 3200x1800                  11
Touchscreen 1366x768                              11
IPS Panel 4K Ultra HD / Touchscreen 3840x2160     10
4K Ultra HD / Touchscreen 3840x2160                7
Touchscreen 2560x1440                              6
IPS Panel Quad HD+ / Touchscreen 3200x1800         6
IPS Panel 4K Ultra HD 3840x2160                    5
IPS Panel Retina Display 2560x1600                 5
Touchscreen 2256x1504                              5
1440x900                                           4
IPS Panel Touchscreen 2560x1440                    4
IPS Panel Retina Display 2304x1440                 4
IPS Panel 1366x768           

In [ ]:
df['r_ancho'] = df['ScreenResolution'].str.extract(r'(\d+)x\d+').astype(int)
df['r_alto'] = df['ScreenResolution'].str.extract(r'\d+x(\d+)').astype(int)
df['r_prop'] = df['r_ancho'] / df['r_alto']
df['definition'] = df['r_ancho'] / df['Inches']

df['is_ips_panel'] = df['ScreenResolution'].str.contains(r'ips\s*panel', case=False)
df['is_touchscreen'] = df['ScreenResolution'].str.contains(r'touchscreen', case=False)
# df['is_retina_display'] = df['ScreenResolution'].str.contains(r'retina\s*display', case=False) # Solo para Apple, descarto

In [ ]:
df['Cpu'].value_counts()

Cpu
Intel Core i5 7200U 2.5GHz              124
Intel Core i7 7700HQ 2.8GHz             105
Intel Core i7 7500U 2.7GHz               97
Intel Core i5 8250U 1.6GHz               52
Intel Core i7 8550U 1.8GHz               47
                                       ... 
Intel Core i3 6006U 2.2GHz                1
Intel Atom Z8350 1.92GHz                  1
Intel Core i5 7200U 2.50GHz               1
AMD A6-Series 7310 2GHz                   1
Intel Pentium Dual Core N4200 1.1GHz      1
Name: count, Length: 107, dtype: int64

In [ ]:
df['ghz'] = df['Cpu'].transform(lambda x: x.split(' ')[-1].replace('GHz', '')).astype(float)

df['is_intel'] = df['Cpu'].str.contains(r'intel', case=False) # Separo entre Intel y AMD

In [ ]:
def cpu_categoria(texto):
    texto = texto.lower()
    if 'core i3' in texto:
        return 3
    if 'core i5' in texto:
        return 4
    if 'core i7' in texto:
        return 4
    if 'core i9' in texto:
        return 5
    if 'pentium dual' in texto:
        return 2
    if 'pentium quad' in texto:
        return 2
    if 'celeron dual' in texto:
        return 2
    if 'celeron quad' in texto:
        return 2
    if 'core m' in texto:
        return 1
    if 'atom' in texto:
        return 1
    if 'xeon' in texto:
        return 5
    if 'ryzen' in texto:
        return 4
    if 'amd a' in texto:
        return 3
    if 'amd e' in texto:
        return 3
    else:
        return 3
    
    
df['cpu_cat'] = df['Cpu'].transform(cpu_categoria)
df['cpu_cat'].value_counts(dropna=False)

cpu_cat
4    659
3    132
2     94
1     25
5      2
Name: count, dtype: int64

In [ ]:
def cpu_modelo(texto):
    modelo = re.search(r'([a-zA-Z]*\d{3,5}[a-zA-Z]*)', texto)
    if not modelo:
        modelo = re.search(r'(\d+Y\d+)', texto)
        
    if modelo:
        return modelo.group(0)
    
    return ''

df['cpu_model'] = df['Cpu'].transform(cpu_modelo)
df['cpu_model'].unique()

array(['6006U', '6700HQ', '7500U', '', '7300U', '7200U', '7100U',
       '7700HQ', '7600U', '8550U', 'N4200', '6500U', '8650U', 'N3710',
       'N3060', 'Z8300', '7300HQ', '6200U', '8250U', '1700', '7820HK',
       'N3700', '9420', '7560U', '6600U', '3855U', 'Z8550', 'N3350',
       'Z8350', '9830P', '6820HK', '7Y75', 'N3050', '9220', '6100U',
       '9720P', '6300U', '7110', '7Y30', '7130U', '6Y54', '6Y75', 'N3450',
       '6300HQ', '6Y30', '7440HQ', '9410', '6820HQ', '3205U', '9000e',
       '7Y54', '1505M', '7820HQ', '7410', '6110', '7660U', '6560U',
       '6440HQ', '9600P', '9000', '7Y57', 'N3160', '6920HQ', '6260U',
       '4405Y', '9620P', '1600', '1535M', '4405U', '7310'], dtype=object)

In [ ]:
def cpu_year_intel(texto):
    digito = re.search(r'\d', texto)
    if digito:
        digito = int(digito.group(0))
        if digito > 5:
            return digito + 9
        
def cpu_year_amd(texto):
    texto = texto if texto else ''
    digito = re.search(r'^\D*(\d)', texto)
    if digito:
        digito = int(digito.group(0))
        if digito > 5:
            return digito + 8

In [ ]:
df.loc[(df['is_intel']) & ((df['cpu_cat'] == 3) | (df['cpu_cat'] == 4)), 'cpu_year'] = df.loc[(df['is_intel']) & ((df['cpu_cat'] == 3) | (df['cpu_cat'] == 4)), 'cpu_model'].transform(cpu_year_intel)
df.loc[~df['is_intel'], 'cpu_year'] = df.loc[~df['is_intel'], 'cpu_model'].apply(cpu_year_amd)
df.loc[(~df['cpu_model'].isna()) & (df['cpu_year'].isna()), 'cpu_year'] = 15
df['cpu_year'].value_counts(dropna=False)

cpu_year
16.0    418
15.0    358
17.0    134
14.0      2
Name: count, dtype: int64

In [ ]:
def cpu_letra(texto):
    letra = re.findall(r'\D+', texto)
    if letra:
        return letra[0]
    else:
        return np.nan

In [ ]:
df['cpu_letra'] = df['cpu_model'].apply(cpu_letra)

otras_cpu_letra = df['cpu_letra'].unique()[df['cpu_letra'].value_counts(dropna=False, sort=False) < 5]

df.loc[df['cpu_letra'].isin(otras_cpu_letra), 'cpu_letra'] = np.nan
df['cpu_letra'].value_counts(dropna=False)

cpu_letra
U      550
HQ     177
N       86
NaN     51
Y       19
Z       10
P       10
HK       9
Name: count, dtype: int64

In [ ]:
df['Ram'].value_counts()

Ram
8GB     434
4GB     267
16GB    136
6GB      24
2GB      20
12GB     19
32GB     10
64GB      1
24GB      1
Name: count, dtype: int64

In [ ]:
df['ram_gb'] = df['Ram'].str.extract(r'(\d+)').astype(int)

df['ram_gb'].value_counts()

ram_gb
8     434
4     267
16    136
6      24
2      20
12     19
32     10
64      1
24      1
Name: count, dtype: int64

In [ ]:
df['Memory'].value_counts()

Memory
256GB SSD                        282
1TB HDD                          152
500GB HDD                         92
512GB SSD                         83
128GB SSD +  1TB HDD              67
128GB SSD                         54
256GB SSD +  1TB HDD              52
32GB Flash Storage                33
1TB SSD                           12
64GB Flash Storage                11
2TB HDD                            8
512GB SSD +  1TB HDD               8
256GB Flash Storage                7
256GB SSD +  2TB HDD               6
16GB Flash Storage                 6
1.0TB Hybrid                       5
32GB SSD                           5
128GB Flash Storage                4
180GB SSD                          3
16GB SSD                           3
1TB SSD +  1TB HDD                 2
512GB SSD +  2TB HDD               2
256GB SSD +  256GB SSD             1
128GB SSD +  2TB HDD               1
512GB SSD +  512GB SSD             1
64GB Flash Storage +  1TB HDD      1
64GB SSD                       

In [ ]:
def memory(lista):
    tipo = {'SSD': 0, 'HDD': 0, 'Flash': 0, 'Hybrid': 0}
    for elemento in lista:
        numero = float(re.match(r'.*(\d+)', elemento).group(0))
        if 'TB' in elemento:
            numero *= 1024
        for key in tipo.keys():
            if key in elemento:
                tipo[key] += numero
    
    return pd.Series(tipo)

In [ ]:
df['drives'] = df['Memory'].str.split('+')

df['n_drives'] = df['drives'].str.len()

In [ ]:
df[['ssd', 'hdd', 'flash', 'hybrid']] = df['drives'].apply(memory)
df['mem'] = df['ssd'] + df['hdd'] + df['flash'] + df['hybrid']

In [ ]:
df['Gpu'].value_counts()

Gpu
Intel HD Graphics 620       185
Intel HD Graphics 520       125
Intel UHD Graphics 620       52
Nvidia GeForce GTX 1050      48
Nvidia GeForce 940MX         31
                           ... 
AMD Radeon RX 540             1
Nvidia Quadro M2000M          1
Nvidia GeForce GTX 940M       1
AMD Radeon R5 520             1
Nvidia GeForce GTX 1070M      1
Name: count, Length: 93, dtype: int64

In [ ]:
def nvidia_categoria(texto):
    texto = texto.lower()
    if 'quadro' in texto or 'sli' in texto:
        return 4
    if '70' in texto or '80' in texto:
        return 3
    if '50' in texto or '60' in texto:
        return 2
    else:
        return 1
    
def amd_categoria(texto):
    texto = texto.lower()
    if 'pro' in texto:
        return 4
    if '60' in texto or '80' in texto:
        return 3
    if '40' in texto or '50' in texto:
        return 2
    else:
        return 1

In [ ]:
df.loc[df['Gpu'].str.contains(r'nvidia', case=False), 'gpu_brand'] = 2
df.loc[df['Gpu'].str.contains(r'amd', case=False), 'gpu_brand'] = 1
df.loc[df['Gpu'].str.contains(r'r\d$|r\d\sg', case=False), 'gpu_brand'] = 0
df['gpu_brand'] = df['gpu_brand'].fillna(0)

In [ ]:
df.loc[df['gpu_brand'] == 2, 'gpu_cat'] = df.loc[df['gpu_brand'] == 2, 'Gpu'].apply(nvidia_categoria)

In [ ]:
df.loc[df['gpu_brand'] == 1, 'gpu_cat'] = df.loc[df['gpu_brand'] == 1, 'Gpu'].apply(amd_categoria)

In [ ]:
df['gpu_cat'] = df['gpu_cat'].fillna(0)

In [ ]:
def gpu_letra(texto):
    if re.findall(r'MX*\d\d+|\d\d+MX*', texto):
        return 1
    if re.findall(r'\d\d+\s*Ti', texto):
        return 2
    return 0

In [ ]:
df.loc[df['gpu_brand'] != 0, 'gpu_letra'] = df.loc[df['gpu_brand'] != 0, 'Gpu'].apply(gpu_letra)
df['gpu_letra'] = df['gpu_letra'].fillna(0)

In [ ]:
df['OpSys'].value_counts()

OpSys
Windows 10      741
Linux            48
No OS            44
Windows 7        29
Chrome OS        24
macOS            11
Windows 10 S      7
Mac OS X          6
Android           2
Name: count, dtype: int64

In [ ]:
os_map = {
    'macOS': 6,
    'Windows 10': 5,
    'Windows 10 S': 4,
    'Windows 7': 3,
    'Linux': 3,
    'Chrome OS': 2,
    'Mac OS X': 2,
    'No OS': 1,
    'Android': 1
}

df['os_ordinal'] = df['OpSys'].map(os_map).fillna(5)
df['os_ordinal'].value_counts()

os_ordinal
5    741
3     77
1     46
2     30
6     11
4      7
Name: count, dtype: int64

In [ ]:
df['Weight'].value_counts()

Weight
2.2kg     91
2.1kg     40
2.4kg     31
2.5kg     29
2.3kg     27
          ..
0.91kg     1
2.15kg     1
2.54kg     1
1.18kg     1
4.33kg     1
Name: count, Length: 165, dtype: int64

In [ ]:
df['weight'] = df['Weight'].str.replace('kg', '').astype(float)
df['density'] = df['Inches'] / df['weight']

In [ ]:
df.corr(numeric_only=True)['Price_in_euros']

Inches             0.071043
Price_in_euros     1.000000
company_ordinal    0.106056
type_ordinal       0.486203
product_len       -0.300375
r_ancho            0.542660
r_alto             0.543591
r_prop            -0.106739
definition         0.459374
is_ips_panel       0.267188
is_touchscreen     0.218089
ghz                0.427116
is_intel           0.176014
cpu_cat            0.487096
cpu_year           0.092929
ram_gb             0.738922
n_drives           0.307600
ssd                0.669151
hdd               -0.076017
flash             -0.042607
hybrid            -0.037336
mem                0.184000
gpu_brand          0.287714
gpu_cat            0.453401
gpu_letra          0.030688
os_ordinal         0.221957
weight             0.197086
density            0.005544
Name: Price_in_euros, dtype: float64

*Por aquí estaba en 260 de score, pero con la ayuda de está función hecha por ChatGPT conseguí rascar 10 puntos extras de score.*

In [ ]:
def extract_product_features(df, company_col='Company', product_col='Product'):
    df = df.copy()
    name = (df[company_col].fillna('') + ' ' + df[product_col].fillna('')).str.lower()

    # ---------- PRODUCT LINE ----------
    product_lines = [
        'thinkpad', 'ideapad', 'xps', 'inspiron', 'latitude', 'precision',
        'elitebook', 'probook', 'pavilion', 'spectre', 'zbook',
        'macbook', 'vivobook', 'zenbook', 'rog', 'predator',
        'aspire', 'swift', 'yoga', 'matebook', 'omen', 'alienware',
        'blade', 'chromebook'
    ]

    def find_product_line(text):
        for pl in product_lines:
            if pl in text:
                return pl
        return 'other'

    df['product_line'] = name.apply(find_product_line)

    # ---------- FLAGS ----------
    gaming_kw = ['gaming', 'rog', 'omen', 'alienware', 'predator', 'blade', 'nitro', 'legion', 'gt', 'gs', 'ge']
    workstation_kw = ['workstation', 'zbook', 'precision', 'thinkpad p', 'quadro', 'firepro']
    business_kw = ['thinkpad', 'elitebook', 'probook', 'latitude', 'precision', 'zbook']

    df['is_gaming'] = name.apply(lambda x: int(any(k in x for k in gaming_kw)))
    df['is_workstation'] = name.apply(lambda x: int(any(k in x for k in workstation_kw)))
    df['is_business'] = name.apply(lambda x: int(any(k in x for k in business_kw)))

    # ---------- PRODUCT TIER ----------
    def product_tier(row):
        if row['is_workstation']:
            return 'workstation'
        if row['is_gaming']:
            return 'gaming'
        if row['product_line'] in ['macbook', 'xps', 'spectre', 'zenbook', 'blade']:
            return 'premium'
        if row['product_line'] in ['ideapad', 'aspire', 'chromebook', 'stream']:
            return 'budget'
        return 'mid'

    df['product_tier'] = df.apply(product_tier, axis=1)

    # ---------- MODEL SERIES NUMBER ----------
    def extract_series(text):
        nums = re.findall(r'\b\d{3,4}\b', text)
        return int(nums[0]) if nums else np.nan

    df['model_series_number'] = name.apply(extract_series)

    # ---------- SPECS IN NAME ----------
    specs_kw = [
        r'i[3579]', r'ryzen', r'\d+gb', r'\d+tb',
        r'geforce', r'radeon', r'ssd', r'hdd'
    ]

    def count_specs(text):
        return sum(bool(re.search(k, text)) for k in specs_kw)

    df['specs_in_name'] = name.apply(count_specs)

    return df

In [ ]:
df = extract_product_features(df, 'Company', 'Product')
df['model_series_number'] = df['model_series_number'].fillna(df['model_series_number'].median())

In [ ]:
df.columns

Index(['Company', 'Product', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu',
       'Ram', 'Memory', 'Gpu', 'OpSys', 'Weight', 'Price_in_euros',
       'company_ordinal', 'type_ordinal', 'product_len', 'r_ancho', 'r_alto',
       'r_prop', 'definition', 'is_ips_panel', 'is_touchscreen', 'ghz',
       'is_intel', 'cpu_cat', 'cpu_model', 'cpu_year', 'cpu_letra', 'ram_gb',
       'drives', 'n_drives', 'ssd', 'hdd', 'flash', 'hybrid', 'mem',
       'gpu_brand', 'gpu_cat', 'gpu_letra', 'os_ordinal', 'weight', 'density',
       'product_line', 'is_gaming', 'is_workstation', 'is_business',
       'product_tier', 'model_series_number', 'specs_in_name'],
      dtype='object')

In [ ]:
df = df.drop(columns=['Company', 'Product', 'TypeName', 'ScreenResolution', 'Cpu', 'Ram', 'Memory', 'Gpu', 'OpSys', 'Weight', 'r_alto', 'cpu_model', 'drives'])

### 2.3 Definir X e y

In [ ]:
X = pd.get_dummies(df.drop(['Price_in_euros'], axis=1))
y = df['Price_in_euros'].copy()
X.shape

(912, 68)

In [ ]:
X.columns

Index(['Inches', 'company_ordinal', 'type_ordinal', 'product_len', 'r_ancho',
       'r_prop', 'definition', 'is_ips_panel', 'is_touchscreen', 'ghz',
       'is_intel', 'cpu_cat', 'cpu_year', 'ram_gb', 'n_drives', 'ssd', 'hdd',
       'flash', 'hybrid', 'mem', 'gpu_brand', 'gpu_cat', 'gpu_letra',
       'os_ordinal', 'weight', 'density', 'is_gaming', 'is_workstation',
       'is_business', 'model_series_number', 'specs_in_name', 'cpu_letra_HK',
       'cpu_letra_HQ', 'cpu_letra_N', 'cpu_letra_P', 'cpu_letra_U',
       'cpu_letra_Y', 'cpu_letra_Z', 'product_line_alienware',
       'product_line_aspire', 'product_line_blade', 'product_line_chromebook',
       'product_line_elitebook', 'product_line_ideapad',
       'product_line_inspiron', 'product_line_latitude',
       'product_line_macbook', 'product_line_matebook', 'product_line_omen',
       'product_line_other', 'product_line_pavilion', 'product_line_precision',
       'product_line_predator', 'product_line_probook', 'product_line_

In [ ]:
y.shape

(912,)

### 2.4 Dividir X_train, X_test, y_train, y_test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 42)

In [ ]:
X_train

,Inches,company_ordinal,type_ordinal,product_len,r_ancho,r_prop,definition,is_ips_panel,is_touchscreen,ghz,is_intel,cpu_cat,cpu_year,ram_gb,n_drives,ssd,hdd,flash,hybrid,mem,gpu_brand,gpu_cat,gpu_letra,os_ordinal,weight,density,is_gaming,is_workstation,is_business,model_series_number,specs_in_name,cpu_letra_HK,cpu_letra_HQ,cpu_letra_N,cpu_letra_P,cpu_letra_U,cpu_letra_Y,cpu_letra_Z,product_line_alienware,product_line_aspire,product_line_blade,product_line_chromebook,product_line_elitebook,product_line_ideapad,product_line_inspiron,product_line_latitude,product_line_macbook,product_line_matebook,product_line_omen,product_line_other,product_line_pavilion,product_line_precision,product_line_predator,product_line_probook,product_line_rog,product_line_spectre,product_line_swift,product_line_thinkpad,product_line_vivobook,product_line_xps,product_line_yoga,product_line_zbook,product_line_zenbook,product_tier_budget,product_tier_gaming,product_tier_mid,product_tier_premium,product_tier_workstation
laptop_ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1118,17.3,5,6,8,1920,1.777778,110.982659,True,False,2.6,True,4,15.0,8,1,0.0,1024.0,0.0,0.0,1024.0,1.0,4.0,1.0,3,3.00,5.766667,0,1,1,850.0,0,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True
153,15.6,6,5,13,1920,1.777778,123.076923,False,False,2.8,True,4,16.0,16,1,512.0,0.0,0.0,0.0,512.0,2.0,2.0,0.0,5,2.56,6.093750,0,0,0,5577.0,0,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
275,13.3,10,4,11,2560,1.600000,192.481203,True,False,2.9,True,4,15.0,8,1,512.0,0.0,0.0,0.0,512.0,0.0,0.0,0.0,6,1.37,9.708029,0,0,0,850.0,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
1100,14.0,5,3,13,1920,1.777778,137.142857,False,False,2.3,True,4,15.0,4,1,0.0,500.0,0.0,0.0,500.0,0.0,0.0,0.0,3,1.54,9.090909,0,0,1,840.0,0,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
131,17.3,6,3,13,1920,1.777778,110.982659,False,False,1.8,True,4,17.0,16,2,256.0,2048.0,0.0,0.0,2304.0,1.0,1.0,0.0,5,2.80,6.178571,0,0,0,5770.0,0,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
578,14.0,5,3,30,1366,1.778646,97.571429,False,False,1.6,True,2,15.0,8,1,0.0,2048.0,0.0,0.0,2048.0,0.0,0.0,0.0,5,1.94,7.216495,0,0,0,850.0,2,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
996,15.6,7,3,17,1920,1.777778,123.076923,False,False,3.6,False,3,17.0,6,1,256.0,0.0,0.0,0.0,256.0,1.0,1.0,0.0,5,2.20,7.090909,0,0,0,320.0,0,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
770,12.5,6,4,13,1920,1.777778,153.600000,False,False,2.8,True,4,16.0,16,1,256.0,0.0,0.0,0.0,256.0,0.0,0.0,0.0,5,1.18,10.5

In [ ]:
y_train

laptop_ID
1118    2899.00
153     1249.26
275     1958.90
1100    1030.99
131     1396.00
         ...   
578      389.00
996      549.00
770     1859.00
407      306.00
418     1943.00
Name: Price_in_euros, Length: 729, dtype: float64

## 3. Procesado de datos

Nuestro target es la columna `Price_in_euros`

In [ ]:
from sklearn.preprocessing import RobustScaler

X_scaled = X.copy()

scaler = RobustScaler()
X_scaled[X_scaled.columns] = scaler.fit_transform(X)

In [ ]:
X_scaled.describe()

,Inches,company_ordinal,type_ordinal,product_len,r_ancho,r_prop,definition,is_ips_panel,is_touchscreen,ghz,is_intel,cpu_cat,cpu_year,ram_gb,n_drives,ssd,hdd,flash,hybrid,mem,gpu_brand,gpu_cat,gpu_letra,os_ordinal,weight,density,is_gaming,is_workstation,is_business,model_series_number,specs_in_name,cpu_letra_HK,cpu_letra_HQ,cpu_letra_N,cpu_letra_P,cpu_letra_U,cpu_letra_Y,cpu_letra_Z,product_line_alienware,product_line_aspire,product_line_blade,product_line_chromebook,product_line_elitebook,product_line_ideapad,product_line_inspiron,product_line_latitude,product_line_macbook,product_line_matebook,product_line_omen,product_line_other,product_line_pavilion,product_line_precision,product_line_predator,product_line_probook,product_line_rog,product_line_spectre,product_line_swift,product_line_thinkpad,product_line_vivobook,product_line_xps,product_line_yoga,product_line_zbook,product_line_zenbook,product_tier_budget,product_tier_gaming,product_tier_mid,product_tier_premium,product_tier_workstation
count,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.00000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.00000,912.000000,912.000000,912.00000,912.000000
mean,-0.386513,0.068348,0.404605,0.425621,-0.074029,-28.698246,0.159609,0.275219,0.143640,-0.237354,-0.046053,-0.430921,-0.250000,0.065789,0.157895,0.220549,0.399744,5.192982,6.171053,0.136930,0.365132,0.760965,0.273026,-0.464912,-0.016127,0.158385,0.141447,0.025219,0.235746,696.952851,0.282895,0.009868,0.194079,0.094298,0.010965,-0.39693,0.020833,0.010965,0.012061,0.046053,0.006579,0.020833,0.043860,0.080044,0.094298,0.042763,0.018640,0.002193,0.012061,0.316886,0.006579,0.007675,0.002193,0.055921,0.031798,0.010965,0.006579,0.073465,0.020833,0.031798,0.024123,0.012061,0.019737,0.14693,0.141447,-0.394737,0.08114,0.025219
std,0.897949,0.579633,0.993779,1.324124,0.975115,171.353554,1.441901,0.446870,0.350917,0.573573,0.209714,0.787978,0.697337,1.261197,0.364842,0.759509,0.504052,30.479484,77.461178,0.613389,0.452959,1.064284,0.499194,1.116131,0.821563,0.781920,0.348674,0.156877,0.424697,1878.039895,0.838831,0.098903,0.395707,0.292404,0.104195,0.48953,0.142905,0.104195,0.109220,0.209714,0.080888,0.142905,0.204895,0.271510,0.292404,0.202434,0.135325,0.046804,0.109220,0.465518,0.080888,0.087321,0.046804,0.229895,0.175559,0.104195,0.080888,0.261041,0.142905,0.175559,0.153515,0.109220,0.139171,0.35423,0.348674,0.489062,0.27320,0.156877
min,-3.437500,-1.333333,-2.000000,-1.166667,-1.111334,-1280.000000,-1.686432,0.000000,0.000000,-1.777778,-1.000000,-3.000000,-2.000000,-1.500000,0.000000,-0.500000,0.000000,0.000000,0.000000,-0.640625,0.000000,0.000000,0.000000,-4.000000,-1.666667,-1.627879,0.000000,0.000000,0.000000,-740.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,-1.000000,0.00000,0.000000
25%,-1.000000,-0.333333,0.000000,-0.333333,-1.000000,0.000000,-0.462315,0.000000,0.000000,-0.777778,0.000000,-1.000000,-1.000000,-1.000000,0.000000,-0.500000,0.000000,0.000000,0.000000,-0.317708,0.000000,0.000000,0.000000,0.000000,-0.679012,-0.330517,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.00000,0.000000,0.00000

In [ ]:
X.describe()

,Inches,company_ordinal,type_ordinal,product_len,r_ancho,r_prop,definition,ghz,cpu_cat,cpu_year,ram_gb,n_drives,ssd,hdd,flash,hybrid,mem,gpu_brand,gpu_cat,gpu_letra,os_ordinal,weight,density,is_gaming,is_workstation,is_business,model_series_number,specs_in_name
count,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000,912.000000
mean,14.981579,5.205044,3.404605,15.553728,1883.096491,1.771550,127.252334,2.286382,3.569079,15.750000,8.263158,1.157895,184.460526,409.337719,5.192982,6.171053,605.162281,0.730263,0.760965,0.273026,4.535088,2.026937,7.951484,0.141447,0.025219,0.235746,1546.952851,0.282895
std,1.436719,1.738900,0.993779,7.944744,486.095047,0.037186,37.720405,0.516215,0.787978,0.697337,5.044788,0.364842,194.434209,516.149265,30.479484,77.461178,471.082514,0.905917,1.064284,0.499194,1.116131,0.665466,1.869430,0.348674,0.156877,0.424697,1878.039895,0.838831
min,10.100000,1.000000,1.000000,6.000000,1366.000000,1.500000,78.959538,0.900000,1.000000,14.000000,2.000000,1.000000,0.000000,0.000000,0.000000,0.000000,8.000000,0.000000,0.000000,0.000000,1.000000,0.690000,3.680851,0.000000,0.000000,0.000000,110.000000,0.000000
25%,14.000000,4.000000,3.000000,11.000000,1421.500000,1.777778,110.982659,1.800000,3.000000,15.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.000000,256.000000,0.000000,0.000000,0.000000,5.000000,1.490000,6.782609,0.000000,0.000000,0.000000,850.000000,0.000000
50%,15.600000,5.000000,3.000000,13.000000,1920.000000,1.777778,123.076923,2.500000,4.000000,16.000000,8.000000,1.000000,128.000000,0.000000,0.000000,0.000000,500.000000,0.000000,0.000000,0.000000,5.000000,2.040000,7.572816,0.000000,0.000000,0.000000,850.000000,0.000000
75%,15.600000,7.000000,4.000000,17.000000,1920.000000,1.777995,137.142857,2.700000,4.000000,16.000000,8.000000,1.000000,256.000000,1024.000000,0.000000,0.000000,1024.000000,2.000000,1.000000,0.000000,5.000000,2.300000,9.173428,0.000000,0.000000,0.000000,850.000000,0.000000
max,18.400000,10.000000,6.000000,45.000000,3840.000000,1.778646,307.200000,3.600000,5.000000,17.000000,64.000000,2.000000,1024.000000,2048.000000,512.000000,1024.000000,2560.000000,2.000000,4.000000,2.000000,6.000000,4.700000,16.419753,1.000000,1.000000,1.000000,9420.000000,4.000000


In [ ]:
base_model = {'lr': LinearRegression(),
              'kn': KNeighborsRegressor(),
              'rf': RandomForestRegressor(max_depth=5),
              'xb': XGBRegressor(max_depth=5),
              'lb': LGBMRegressor(max_depth=5, verbosity=-1),
}

for k, m in base_model.items():
    print(f'------{k}------')
    print(np.mean(cross_val_score(m, X, y, scoring='neg_root_mean_squared_error')))
    print()

------lr------
-283.7686381871264

------kn------
-402.0964318435772

------rf------
-296.5685737456409

------xb------
-231.48006728554984

------lb------
-253.52686505449915



In [ ]:
base_model = {'lr': LinearRegression(),
              'kn': KNeighborsRegressor(),
              'rf': RandomForestRegressor(max_depth=5),
              'xb': XGBRegressor(max_depth=5),
              'lb': LGBMRegressor(max_depth=5, verbosity=-1),
}

for k, m in base_model.items():
    print(f'------{k}------')
    print(np.mean(cross_val_score(m, X_scaled, y, scoring='neg_root_mean_squared_error')))
    print()

------lr------
-283.7686381871314

------kn------
-348.81332789077516

------rf------
-298.3149742710822

------xb------
-231.48006728554984

------lb------
-254.19002994714793



In [ ]:
base_model['rf'].fit(X, y)

pd.DataFrame(base_model['rf'].feature_importances_, index=base_model['rf'].feature_names_in_).sort_values(0, ascending=False)

,0
ram_gb,0.546523
cpu_cat,0.101763
gpu_cat,0.058433
density,0.038692
type_ordinal,0.037227
...,...
product_line_spectre,0.000000
product_line_vivobook,0.000000
product_line_swift,0.000000
product_line_predator,0.000000


In [ ]:
model = XGBRegressor(n_estimators=1000)

param_grid = {
    'n_estimators': [200, 300, 400, 600, 800],
    'learning_rate': [0.001, 0.1, 0.05, 0.1, 0.2],
    'max_depth': [2, 3, 4, 5, 7, 9],
    'subsample': [0.6, 0,7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0]
}

grid_model = GridSearchCV(model, param_grid, scoring='neg_root_mean_squared_error', verbose=1)

grid_model.fit(X, y)

Fitting 5 folds for each of 4500 candidates, totalling 22500 fits


c:\Users\phbas\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
3750 fits failed out of a total of 22500.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3750 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\phbas\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\phbas\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py", line 774, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "c:\Users\phbas\AppData\Local\Programs\Python\Python312\Lib\site

GridSearchCV(estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=False, eval_metric=None,
                                    feature_types=None, feature_weights=None,
                                    gamma=None, grow_policy=None,
                                    importance_type=None,
                                    interaction_constraints=None,
                                    lear...
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=1000,
                                    n_jobs=None, num_parallel_tree=None, ...),
             param_grid={'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
                         'learning_rate': [0.001, 0.1, 0.05, 0.1, 0.2],
                         'max_depth': [2, 3, 4, 5, 7, 9],
                         'n_estimators': [200, 300, 400, 600, 800],
                         'subsample': [0.6, 0, 7, 0.8, 0.9, 1.0]},
             scoring='neg_root_mean_squared_error', verbose=1)

-----------------------------------------------------------------------------------------------------------------

In [ ]:
pd.DataFrame(grid_model.cv_results_).sort_values('rank_test_score').head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_colsample_bytree,param_learning_rate,param_max_depth,param_n_estimators,param_subsample,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
2385,0.137925,0.001471,0.008407,4.902908e-04,0.8,0.1,3,400,0.8,"{'colsample_bytree': 0.8, 'learning_rate': 0.1...",-278.672236,-187.511378,-243.943720,-215.911851,-161.224867,-217.452810,41.255380,1
2025,0.146733,0.005009,0.008008,1.933397e-06,0.8,0.1,3,400,0.8,"{'colsample_bytree': 0.8, 'learning_rate': 0.1...",-278.672236,-187.511378,-243.943720,-215.911851,-161.224867,-217.452810,41.255380,1
2031,0.204586,0.003777,0.008007,2.132481e-07,0.8,0.1,3,600,0.8,"{'colsample_bytree': 0.8, 'learning_rate': 0.1...",-282.173110,-188.626099,-245.609835,-212.593866,-160.340516,-217.868685,42.654597,3
2391,0.204386,0.002229,0.008007,6.330883e-04,0.8,0.1,3,600,0.8,"{'colsample_bytree': 0.8, 'learning_rate': 0.1...",-282.173110,-188.626099,-245.609835,-212.593866,-160.340516,-217.868685,42.654597,3
2397,0.267042,0.001602,0.008007,1.784161e-07,0.8,0.1,3,800,0.8,"{'colsample_bytree': 0.8, 'learning_rate': 0.1...",-280.955100,-189.808679,-247.336774,-212.661162,-160.290262,-218.210396,42.373193,5


In [ ]:
model = XGBRegressor(n_estimators=1000)

param_grid = {
    'n_estimators': [300, 400, 500],
    'learning_rate': [0.075, 0.1, 0.15],
    'max_depth': [3],
    'subsample': [0.75, 0.8, 0.85],
    'colsample_bytree': [0.75, 0.8, 0.85]
}

grid_model = GridSearchCV(model, param_grid, scoring='neg_root_mean_squared_error', verbose=1)

grid_model.fit(X, y)

Fitting 5 folds for each of 81 candidates, totalling 405 fits


GridSearchCV(estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=False, eval_metric=None,
                                    feature_types=None, feature_weights=None,
                                    gamma=None, grow_policy=None,
                                    importance_type=None,
                                    interaction_constraints=None,
                                    lear...
                                    max_depth=None, max_leaves=None,
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=1000,
                                    n_jobs=None, num_parallel_tree=None, ...),
             param_grid={'colsample_bytree': [0.75, 0.8, 0.85],
                         'learning_rate': [0.075, 0.1, 0.15], 'max_depth': [3],
                         'n_estimators': [300, 400, 500],
                         'subsample': [0.75, 0.8, 0.85]},
             scoring='neg_root_mean_squared_error', verbose=1)

In [ ]:
grid_df = pd.DataFrame(grid_model.cv_results_).sort_values('rank_test_score')
grid_df.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_colsample_bytree,param_learning_rate,param_max_depth,param_n_estimators,param_subsample,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
22,0.129405,0.000387,0.007228,0.000409,0.75,0.15,3,400,0.8,"{'colsample_bytree': 0.75, 'learning_rate': 0....",-277.763618,-183.787002,-252.173269,-208.107756,-163.362696,-217.038868,42.399302,1
40,0.130358,0.000437,0.007629,0.000495,0.80,0.10,3,400,0.8,"{'colsample_bytree': 0.8, 'learning_rate': 0.1...",-278.672236,-187.511378,-243.943720,-215.911851,-161.224867,-217.452810,41.255380,2
25,0.160094,0.001005,0.007837,0.000375,0.75,0.15,3,500,0.8,"{'colsample_bytree': 0.75, 'learning_rate': 0....",-277.603622,-185.690733,-252.944214,-208.188325,-163.426565,-217.570692,42.169149,3
43,0.158891,0.000722,0.007238,0.000385,0.80,0.10,3,500,0.8,"{'colsample_bytree': 0.8, 'learning_rate': 0.1...",-281.007652,-188.618698,-244.421761,-214.072037,-160.096021,-217.643234,42.187671,4
19,0.099677,0.000764,0.007241,0.000400,0.75,0.15,3,300,0.8,"{'colsample_bytree': 0.75, 'learning_rate': 0....",-278.943639,-183.703180,-252.665904,-211.202761,-163.779724,-218.059042,42.610204,5


In [ ]:
def get_par(index):
    dicc = {}
    for col in grid_df.columns:
        if col.startswith('param_'):
            dicc[col.replace('param_', '')] = grid_df.loc[index, col]
        
    return dicc

In [ ]:
parametros = get_par(22)
parametros.pop('n_estimators')
parametros

{'colsample_bytree': np.float64(0.75),
 'learning_rate': np.float64(0.15),
 'max_depth': np.int64(3),
 'subsample': np.float64(0.8)}

In [ ]:
model = XGBRegressor(**parametros, n_estimators=400, early_stopping_rounds=50)

model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=np.float64(0.75), device=None,
             early_stopping_rounds=50, enable_categorical=False,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=np.float64(0.15),
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=np.int64(3), max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=400, n_jobs=None,
             num_parallel_tree=None, ...)

In [ ]:
model.best_iteration

356

In [ ]:
model_def = XGBRegressor(**parametros)
model_def.fit(X, y)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=np.float64(0.75), device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=np.float64(0.15),
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=np.int64(3), max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, ...)

In [ ]:
print('model_def ->', root_mean_squared_error(y_test, model_def.predict(X_test)))

model_def -> 143.11146968167748


In [ ]:
model_def = XGBRegressor(**parametros, n_estimators=model.best_iteration)
model_def.fit(X, y)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=np.float64(0.75), device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=np.float64(0.15),
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=np.int64(3), max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=356, n_jobs=None,
             num_parallel_tree=None, ...)

In [ ]:
print('model_def ->', root_mean_squared_error(y_test, model_def.predict(X_test)))

model_def -> 84.24031146872382


## 4. Modelado

### 4.1 Baseline de modelos


### 4.2 Sacar métricas, valorar los modelos

Recuerda que en la competición se va a evaluar con la métrica de ``RMSE``.

### 4.3 Optimización (up to you 🫰🏻)

-----------------------------------------------------------------

## Una vez listo el modelo, toca predecir ``test.csv``

**RECUERDA: APLICAR LAS TRANSFORMACIONES QUE HAYAS REALIZADO EN `train.csv` a `test.csv`.**


Véase:
- Estandarización/Normalización
- Eliminación de Outliers
- Eliminación de columnas
- Creación de columnas nuevas
- Gestión de valores nulos
- Y un largo etcétera de técnicas que como Data Scientist hayas considerado las mejores para tu dataset.

## 1. Carga los datos de `test.csv` para predecir.


In [ ]:
df = pd.read_csv("./data/test.csv", index_col=0)
df.head()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
laptop_ID,,,,,,,,,,,
209,Lenovo,Legion Y520-15IKBN,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1060,No OS,2.4kg
1281,Acer,Aspire ES1-531,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4GB,500GB HDD,Intel HD Graphics 400,Linux,2.4kg
1168,Lenovo,V110-15ISK (i3-6006U/4GB/1TB/No,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,Intel HD Graphics 520,No OS,1.9kg
1231,Dell,Inspiron 7579,2 in 1 Convertible,15.6,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,2.191kg
1020,HP,ProBook 640,Notebook,14.0,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.95kg


In [ ]:
df.tail()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
laptop_ID,,,,,,,,,,,
820,MSI,GE72MVR 7RG,Gaming,17.3,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD + 1TB HDD,Nvidia GeForce GTX 1070,Windows 10,2.9kg
948,Toshiba,Tecra Z40-C-12X,Notebook,14.0,IPS Panel Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,4GB,128GB SSD,Intel HD Graphics 520,Windows 10,1.47kg
483,Dell,Precision M5520,Workstation,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,8GB,256GB SSD,Nvidia Quadro M1200,Windows 10,1.78kg
1017,HP,Probook 440,Notebook,14.0,1366x768,Intel Core i5 7200U 2.5GHz,4GB,500GB HDD,Intel HD Graphics 620,Windows 10,1.64kg
421,Asus,ZenBook Flip,2 in 1 Convertible,13.3,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.27kg


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 391 entries, 209 to 421
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           391 non-null    object 
 1   Product           391 non-null    object 
 2   TypeName          391 non-null    object 
 3   Inches            391 non-null    float64
 4   ScreenResolution  391 non-null    object 
 5   Cpu               391 non-null    object 
 6   Ram               391 non-null    object 
 7   Memory            391 non-null    object 
 8   Gpu               391 non-null    object 
 9   OpSys             391 non-null    object 
 10  Weight            391 non-null    object 
dtypes: float64(1), object(10)
memory usage: 36.7+ KB


 ## 2. Replicar el procesado para ``test.csv``

In [ ]:
df['company_ordinal'] = df['Company'].map(brand_map).fillna(6)

In [ ]:
df['type_ordinal'] = df['TypeName'].map(type_map).fillna(3)

In [ ]:
df['product_len'] = df['Product'].str.len()

In [ ]:
df['r_ancho'] = df['ScreenResolution'].str.extract(r'(\d+)x\d+').astype(int)
df['r_alto'] = df['ScreenResolution'].str.extract(r'\d+x(\d+)').astype(int)
df['r_prop'] = df['r_ancho'] / df['r_alto']
df['definition'] = df['r_ancho'] / df['Inches']

df['is_ips_panel'] = df['ScreenResolution'].str.contains(r'ips\s*panel', case=False)
df['is_touchscreen'] = df['ScreenResolution'].str.contains(r'touchscreen', case=False)
# df['is_retina_display'] = df['ScreenResolution'].str.contains(r'retina\s*display', case=False) # Solo para Apple, descarto

In [ ]:
df['ghz'] = df['Cpu'].transform(lambda x: x.split(' ')[-1].replace('GHz', '')).astype(float)

df['is_intel'] = df['Cpu'].str.contains(r'intel', case=False) # Separo entre Intel y AMD

In [ ]:
df['cpu_cat'] = df['Cpu'].transform(cpu_categoria)

In [ ]:
df['cpu_model'] = df['Cpu'].transform(cpu_modelo)

In [ ]:
df.loc[(df['is_intel']) & ((df['cpu_cat'] == 3) | (df['cpu_cat'] == 4)), 'cpu_year'] = df.loc[(df['is_intel']) & ((df['cpu_cat'] == 3) | (df['cpu_cat'] == 4)), 'cpu_model'].transform(cpu_year_intel)
df.loc[~df['is_intel'], 'cpu_year'] = df.loc[~df['is_intel'], 'cpu_model'].apply(cpu_year_amd)
df.loc[(~df['cpu_model'].isna()) & (df['cpu_year'].isna()), 'cpu_year'] = 15

In [ ]:
df['cpu_letra'] = df['cpu_model'].apply(cpu_letra)
df.loc[df['cpu_letra'].isin(otras_cpu_letra), 'cpu_letra'] = np.nan

In [ ]:
df['ram_gb'] = df['Ram'].str.extract(r'(\d+)').astype(int)

In [ ]:
df['drives'] = df['Memory'].str.split('+')
df['n_drives'] = df['drives'].str.len()

In [ ]:
df[['ssd', 'hdd', 'flash', 'hybrid']] = df['drives'].apply(memory)
df['mem'] = df['ssd'] + df['hdd'] + df['flash'] + df['hybrid']

In [ ]:
df.loc[df['Gpu'].str.contains(r'nvidia', case=False), 'gpu_brand'] = 2
df.loc[df['Gpu'].str.contains(r'amd', case=False), 'gpu_brand'] = 1
df.loc[df['Gpu'].str.contains(r'r\d$|r\d\sg', case=False), 'gpu_brand'] = 0
df['gpu_brand'] = df['gpu_brand'].fillna(0)

In [ ]:
df.loc[df['gpu_brand'] == 2, 'gpu_cat'] = df.loc[df['gpu_brand'] == 2, 'Gpu'].apply(nvidia_categoria)

In [ ]:
df.loc[df['gpu_brand'] == 1, 'gpu_cat'] = df.loc[df['gpu_brand'] == 1, 'Gpu'].apply(amd_categoria)

In [ ]:
df['gpu_cat'] = df['gpu_cat'].fillna(0)

In [ ]:
df.loc[df['gpu_brand'] != 0, 'gpu_letra'] = df.loc[df['gpu_brand'] != 0, 'Gpu'].apply(gpu_letra)
df['gpu_letra'] = df['gpu_letra'].fillna(0)

In [ ]:
df['os_ordinal'] = df['OpSys'].map(os_map).fillna(5)

In [ ]:
df['weight'] = df['Weight'].str.replace('kg', '').astype(float)
df['density'] = df['Inches'] / df['weight']

In [ ]:
df = extract_product_features(df, 'Company', 'Product')
df['model_series_number'] = df['model_series_number'].fillna(df['model_series_number'].median())

In [ ]:
df.columns

Index(['Company', 'Product', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu',
       'Ram', 'Memory', 'Gpu', 'OpSys', 'Weight', 'company_ordinal',
       'type_ordinal', 'product_len', 'r_ancho', 'r_alto', 'r_prop',
       'definition', 'is_ips_panel', 'is_touchscreen', 'ghz', 'is_intel',
       'cpu_cat', 'cpu_model', 'cpu_year', 'cpu_letra', 'ram_gb', 'drives',
       'n_drives', 'ssd', 'hdd', 'flash', 'hybrid', 'mem', 'gpu_brand',
       'gpu_cat', 'gpu_letra', 'os_ordinal', 'weight', 'density',
       'product_line', 'is_gaming', 'is_workstation', 'is_business',
       'product_tier', 'model_series_number', 'specs_in_name'],
      dtype='object')

In [ ]:
df = df.drop(columns=['Company', 'Product', 'TypeName', 'ScreenResolution', 'Cpu', 'Ram', 'Memory', 'Gpu', 'OpSys', 'Weight', 'r_alto', 'cpu_model', 'drives'])

In [ ]:
df = pd.get_dummies(df)

In [ ]:
df.columns

Index(['Inches', 'company_ordinal', 'type_ordinal', 'product_len', 'r_ancho',
       'r_prop', 'definition', 'is_ips_panel', 'is_touchscreen', 'ghz',
       'is_intel', 'cpu_cat', 'cpu_year', 'ram_gb', 'n_drives', 'ssd', 'hdd',
       'flash', 'hybrid', 'mem', 'gpu_brand', 'gpu_cat', 'gpu_letra',
       'os_ordinal', 'weight', 'density', 'is_gaming', 'is_workstation',
       'is_business', 'model_series_number', 'specs_in_name', 'cpu_letra_HK',
       'cpu_letra_HQ', 'cpu_letra_N', 'cpu_letra_P', 'cpu_letra_U',
       'cpu_letra_Y', 'cpu_letra_Z', 'product_line_alienware',
       'product_line_aspire', 'product_line_blade', 'product_line_chromebook',
       'product_line_elitebook', 'product_line_ideapad',
       'product_line_inspiron', 'product_line_latitude',
       'product_line_macbook', 'product_line_other', 'product_line_pavilion',
       'product_line_precision', 'product_line_predator',
       'product_line_probook', 'product_line_rog', 'product_line_spectre',
       'product_

In [ ]:
df = df.reindex(columns=X.columns, fill_value=0)

In [ ]:
predictions_submit = {'Price_in_euros': model_def.predict(df)}
predictions_submit

{'Price_in_euros': array([1431.5803 ,  292.20612,  350.5176 , 1143.2244 , 1151.9056 ,
         655.54724,  886.9218 ,  945.5597 , 1072.6608 ,  309.07556,
        2417.4666 , 1382.8315 ,  435.30963, 1510.2094 ,  925.55786,
         665.8429 , 2068.024  , 1272.2437 , 1755.2097 ,  681.94244,
        1274.658  ,  266.95154,  926.948  , 1041.3595 ,  377.13034,
         663.81506,  554.0043 ,  768.72986, 2612.1665 , 1132.5956 ,
        2463.339  ,  377.46368, 1013.4048 , 3128.7488 , 2315.1858 ,
        1565.9885 ,  672.99207, 1423.5493 ,  895.4395 , 1763.7876 ,
         751.17883,  692.40356,  514.2836 , 1138.7473 , 1036.7197 ,
        1109.3364 , 1094.4303 ,  660.7942 ,  600.4226 ,  382.51736,
        1710.1312 ,  681.62744, 1143.6173 ,  415.9934 , 1875.9095 ,
        1767.4548 ,  613.72217,  879.86   ,  877.2837 ,  536.96844,
        2725.567  , 2030.1475 ,  458.0801 , 2096.2234 , 1619.3378 ,
        1513.4358 , 1082.6093 , 1019.77893, 1817.862  , 1979.6968 ,
         909.1195 ,  472.9879 

**¡OJO! ¿Por qué me da error?**

IMPORTANTE:

- SI EL ARRAY CON EL QUE HICISTEIS `.fit()` ERA DE 4 COLUMNAS, PARA `.predict()` DEBEN SER LAS MISMAS
- SI AL ARRAY CON EL QUE HICISTEIS `.fit()` LO NORMALIZASTEIS, PARA `.predict()` DEBÉIS NORMALIZARLO
- TODO IGUAL SALVO **BORRAR FILAS**, EL NÚMERO DE ROWS SE DEBE MANTENER EN ESTE SET, PUES LA PREDICCIÓN DEBE TENER **391 FILAS**, SI O SI

**Entonces, si al cargar los datos de ``train.csv`` usaste `index_col=0`, ¿tendré que hacer lo también para el `test.csv`?**

In [ ]:
# ¿Qué opináis?
# ¿Sí, no?

![wow.jpeg](attachment:wow.jpeg)

## 3. **¿Qué es lo que subirás a Kaggle?**

**Para subir a Kaggle la predicción esta tendrá que tener una forma específica.**

En este caso, la **MISMA** forma que `sample_submission.csv`.

In [ ]:
sample = pd.read_csv("data/sample_submission.csv")

In [ ]:
sample.head()

,laptop_ID,Price_in_euros
0,209,1949.1
1,1281,805.0
2,1168,1101.0
3,1231,1293.8
4,1020,1832.6


In [ ]:
sample.shape

(391, 2)

## 4. Mete tus predicciones en un dataframe llamado ``submission``.

In [ ]:
#¿Cómo creamos la submission?
submission = pd.DataFrame(predictions_submit, index=df.index).reset_index()

In [ ]:
submission.head()

,laptop_ID,Price_in_euros
0,209,1431.580322
1,1281,292.206116
2,1168,350.517609
3,1231,1143.224365
4,1020,1151.905640


In [ ]:
submission.shape

(391, 2)

## 5. Pásale el CHEQUEADOR para comprobar que efectivamente está listo para subir a Kaggle.

In [ ]:
def chequeador(df_to_submit):
    """
    Esta función se asegura de que tu submission tenga la forma requerida por Kaggle.

    Si es así, se guardará el dataframe en un `csv` y estará listo para subir a Kaggle.

    Si no, LEE EL MENSAJE Y HAZLE CASO.

    Si aún no:
    - apaga tu ordenador,
    - date una vuelta,
    - enciendelo otra vez,
    - abre este notebook y
    - leelo todo de nuevo.
    Todos nos merecemos una segunda oportunidad. También tú.
    """
    if df_to_submit.shape == sample.shape:
        if df_to_submit.columns.all() == sample.columns.all():
            if df_to_submit.laptop_ID.all() == sample.laptop_ID.all():
                print("You're ready to submit!")
                df_to_submit.to_csv("submission.csv", index = False) #muy importante el index = False
                urllib.request.urlretrieve("https://www.mihaileric.com/static/evaluation-meme-e0a350f278a36346e6d46b139b1d0da0-ed51e.jpg", "gfg.png")
                img = Image.open("gfg.png")
                img.show()
            else:
                print("Check the ids and try again")
        else:
            print("Check the names of the columns and try again")
    else:
        print("Check the number of rows and/or columns and try again")
        print("\nMensaje secreto del TA: No me puedo creer que después de todo este notebook hayas hecho algún cambio en las filas de `test.csv`. Lloro.")

In [ ]:
chequeador(submission)

You're ready to submit!
